<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>
<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Curso de Hidrología</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Delimitación de una cuenca hidrográfica

## Objetivo
Delimitar una cuenca hidrográfica a partir de un Modelo Digital de Elevación (DEM) y un punto de salida.

---

## Conceptos clave

### ¿Qué es una cuenca hidrográfica?
Una cuenca hidrográfica es el área donde toda el agua escurre hacia un mismo punto de salida.

### ¿Qué es un DEM?
Un DEM (Digital Elevation Model) es una grilla donde cada píxel representa la elevación del terreno.

Es útil para:
- analizar flujo de agua  
- delimitar cuencas  
- calcular pendientes  

# Recurso recomendado:

Mapa interactivo permite rastrear el trayecto de una gota de agua hasta el mar

https://river-runner-global.samlearner.com/

## Antes de empezar

En este curso utilizamos materiales (código y datos) almacenados en un repositorio de GitHub.

### ¿Qué es un repositorio?
Un repositorio es una carpeta en línea que contiene archivos de un proyecto, como:
- código
- datos
- notebooks
- documentación

### ¿Qué significa "clonar un repositorio"?
Clonar un repositorio significa **descargar una copia completa de ese proyecto desde internet a nuestro entorno de trabajo (Google Colab)**.

Esto nos permite:
- acceder a los datos del curso  
- ejecutar los notebooks  
- modificar el código  

En este caso, vamos a clonar el repositorio del curso de hidrología.



In [ ]:
# Clonar el repositorio desde GitHub
!git clone -- https://github.com/paulmunozpauta/Hidrology_Course

Después de ejecutar la celda anterior, se crea una carpeta en nuestro entorno llamada:

**Hidrology_Course**

Esta carpeta contiene todos los archivos del curso.

Ahora debemos entrar a esa carpeta para trabajar con los datos.

In [ ]:
ls

# Entrar a la carpeta del repositorio





In [ ]:
# Entrar a la carpeta del repositorio
%cd Hidrology_Course
!ls

Ahora estamos dentro del repositorio del curso.

Aquí podemos encontrar:

- notebooks (.ipynb)
- carpeta Data/ con los datos
- archivos de apoyo

En este notebook trabajaremos con datos de la Región de Ñuble.

## Instalación de librerías

En esta sección instalamos las librerías necesarias.

### Librerías principales:
- rasterio → manejo de datos raster  
- geopandas → datos vectoriales  
- shapely → geometría  
- pysheds → análisis hidrológico  
- leafmap → mapas interactivos  

# Delimitación de cuenca hidrográfica

Este notebook permite delimitar una cuenca hidrográfica a partir de:

- un DEM (`REGION_NUBLE_1.jp2`)
- un punto de salida definido con coordenadas `lon/lat`


In [ ]:
# Instalar librerías necesarias
!pip -q install leafmap pysheds rasterio geopandas shapely

In [ ]:
# Importar librerías
import os
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd
from pysheds.grid import Grid
import leafmap.foliumap as leafmap
from rasterio.plot import show
from rasterio.enums import Resampling
from rasterio.warp import transform
from rasterio.plot import show
from shapely.geometry import Point
from matplotlib.colors import LinearSegmentedColormap
from shapely.ops import unary_union
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np
from shapely.geometry import Point
from shapely.ops import unary_union


def snap_to_high_accumulation(
    dem_path,
    acc,
    x,
    y,
    search_radius=10
):
    """
    Ajusta un punto al píxel cercano con mayor acumulación de flujo.

    Parámetros
    ----------
    dem_path : str
        Ruta del DEM.

    acc : ndarray
        Matriz de acumulación de flujo.

    x, y : float
        Coordenadas del punto en el CRS del DEM.

    search_radius : int
        Radio de búsqueda en píxeles.

    Retorna
    -------
    snapped_x, snapped_y : float
        Coordenadas ajustadas.

    snapped_row, snapped_col : int
        Índices del píxel ajustado.
    """

    with rasterio.open(dem_path) as src:
        # Abrimos el raster (DEM) con rasterio.
        # "src" contiene información como:
        # - sistema de coordenadas (CRS)
        # - resolución
        # - transformación entre coordenadas y píxeles

        # ---------------------------------------------------------
        # Convertir coordenadas (x, y) a índices del raster
        # ---------------------------------------------------------
        row, col = src.index(x, y)
        # src.index() transforma coordenadas espaciales (ej: lon/lat o metros)
        # a posición en la matriz del raster:
        # - row → fila (vertical)
        # - col → columna (horizontal)
        # Esto permite ubicar el punto dentro del DEM.

        # ---------------------------------------------------------
        # Definir ventana de búsqueda alrededor del punto
        # ---------------------------------------------------------
        row_min = max(0, row - search_radius)
        # Límite inferior de filas (evita salir del raster)

        row_max = min(acc.shape[0], row + search_radius + 1)
        # Límite superior de filas (máximo permitido)

        col_min = max(0, col - search_radius)
        # Límite izquierdo de columnas

        col_max = min(acc.shape[1], col + search_radius + 1)
        # Límite derecho de columnas

        # En conjunto, esto define una ventana cuadrada centrada en el punto:
        # tamaño ≈ (2 * search_radius + 1)

        # ---------------------------------------------------------
        # Extraer acumulación en la ventana
        # ---------------------------------------------------------
        window_acc = acc[row_min:row_max, col_min:col_max]
        # Se recorta la matriz de acumulación solo en esa zona local.
        # Esto reduce el problema a un área pequeña alrededor del punto.

        # ---------------------------------------------------------
        # Verificar que la ventana no esté vacía
        # ---------------------------------------------------------
        if window_acc.size == 0:
            raise ValueError("Ventana de búsqueda vacía.")
        # Esto evita errores si el punto está fuera del raster
        # o en un borde extremo.

        # ---------------------------------------------------------
        # Encontrar el píxel con mayor acumulación
        # ---------------------------------------------------------
        local_idx = np.unravel_index(
            np.nanargmax(window_acc),
            window_acc.shape
        )
        # np.nanargmax() encuentra la posición del valor máximo ignorando NaN.
        # np.unravel_index convierte ese índice en:
        # (fila_local, columna_local) dentro de la ventana.

        # ---------------------------------------------------------
        # Convertir índices locales a globales del raster
        # ---------------------------------------------------------
        snapped_row = row_min + local_idx[0]
        # Ajusta la fila al sistema completo del raster

        snapped_col = col_min + local_idx[1]
        # Ajusta la columna al sistema completo del raster

        # Ahora tenemos la posición exacta del píxel con mayor acumulación
        # dentro de todo el DEM.

        # ---------------------------------------------------------
        # Convertir índices raster a coordenadas espaciales
        # ---------------------------------------------------------
        snapped_x, snapped_y = src.xy(snapped_row, snapped_col)
        # src.xy() hace la operación inversa de src.index():
        # convierte (fila, columna) → coordenadas reales (x, y)
        # Estas coordenadas están en el CRS del DEM.

    # ---------------------------------------------------------
    # Retornar resultados
    # ---------------------------------------------------------
    return snapped_x, snapped_y, snapped_row, snapped_col
    # snapped_x, snapped_y → coordenadas espaciales del nuevo punto
    # snapped_row, snapped_col → ubicación en la matriz del raster


## Visualización del área de estudio

Este mapa sirve para ubicar la zona y elegir el punto de salida.


In [ ]:
# Crear mapa base
m = leafmap.Map(center=[-36.59, -72.08], zoom=7)
m.add_basemap("OpenTopoMap")
m

## Cargar el DEM
El DEM es la base del análisis hidrológico.
Revisamos:
- resolución  
- dimensiones  
- sistema de coordenadas


### DEM oficial Hidrografía Chile (DGA)
https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/hidrosfera/resource/7776d37b-8a9f-4bf9-be75-006b40af0b1b

In [ ]:
dem_path = "Data/Ñuble/REGION_NUBLE_1.jp2"

with rasterio.open(dem_path) as src:
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("CRS:", src.crs)

## Reducir resolución del DEM

Para facilitar el procesamiento, creamos una versión más liviana.

Esto NO cambia el DEM original.

In [ ]:
dem_lowres_path = "Data/Ñuble/dem_lowres.tif"
scale_factor = 8
with rasterio.open(dem_path) as src:
    data = src.read(
        1,
        out_shape=(int(src.height/scale_factor), int(src.width/scale_factor)),
        resampling=Resampling.average
    )

    transform_lowres = src.transform * src.transform.scale(
        (src.width / data.shape[1]),
        (src.height / data.shape[0])
    )

    profile = src.profile.copy()
    profile.update({
        "height": data.shape[0],
        "width": data.shape[1],
        "transform": transform_lowres
    })

    with rasterio.open(dem_lowres_path, "w", **profile) as dst:
        dst.write(data, 1)
    print("DEM lowres guardado en:", dem_lowres_path)

## Visualización inicial del área de estudio

Primera inspección para:

- reconocer la ubicación general de la cuenca,
- identificar ríos y relieve,
- escoger un punto de salida aproximado.




In [ ]:
with rasterio.open(dem_lowres_path) as src:
    dem_plot = src.read(1).astype(float)
    bounds = src.bounds
    crs = src.crs

    if src.nodata is not None:
        dem_plot[dem_plot == src.nodata] = np.nan

dem_plot = np.ma.masked_where(np.isnan(dem_plot), dem_plot)

colors = [
    (0.85, 0.92, 0.75),
    (0.55, 0.75, 0.45),
    (0.85, 0.75, 0.55),
    (0.65, 0.45, 0.30),
    (0.95, 0.95, 0.95)
]

cmap = LinearSegmentedColormap.from_list("custom_terrain", colors, N=256)

fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(
    dem_plot,
    cmap=cmap,
    extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
    origin="upper"
)

ax.set_title("DEM")

# Etiquetas según tipo de CRS
if crs.is_geographic:
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
else:
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Elevación (m)")

plt.show()


## Visualización del DEM

En esta figura se representa el Modelo Digital de Elevación (DEM).

### ¿Qué observamos aquí?
Cada píxel del raster tiene un valor de elevación. Los colores permiten distinguir:

- zonas bajas,
- zonas medias,
- zonas altas.


## Preparación hidrológica del DEM

### Pasos:

1. Rellenar depresiones  
   Se eliminan huecos o "pozos" en el terreno donde el agua quedaría atrapada de forma artificial.

2. Resolver zonas planas  
   Se define hacia dónde fluye el agua en áreas completamente planas donde no hay pendiente clara.

3. Calcular dirección de flujo  
   Se determina hacia qué vecino fluye el agua en cada celda (la dirección de la pendiente).

4. Calcular acumulación  
   Se calcula cuánta agua llega a cada celda, sumando todo el flujo aguas arriba.


In [ ]:
grid = Grid.from_raster(dem_lowres_path)
dem = grid.read_raster(dem_lowres_path)
print("Rellenando depresiones...")
dem_filled = grid.fill_depressions(dem)
print("Resolviendo zonas planas...")
dem_inflated = grid.resolve_flats(dem_filled)
print("Calculando dirección de flujo...")
fdir = grid.flowdir(dem_inflated)
print("Calculando acumulación...")
acc = grid.accumulation(fdir)
print("DEM listo para el análisis hidrológico.")

## Punto de salida

El punto de salida es donde el agua abandona la cuenca.

Debe ubicarse sobre un río.

In [ ]:
# Coordenadas del punto de salida
approximate_lon = -72.25
approximate_lat = -36.6
point = Point(approximate_lon, approximate_lat)
print(f"Punto de salida: lon={approximate_lon}, lat={approximate_lat}")

# Convertir el punto al CRS (Coordinate Reference System) del DEM
Todas las operaciones espaciales deben estar en el mismo sistema de coordenadas (CRS)


In [ ]:
with rasterio.open(dem_lowres_path) as src:
    x_out, y_out = transform("EPSG:4326", src.crs, [approximate_lon], [approximate_lat])
    x_out = x_out[0]
    y_out = y_out[0]

print("Punto en el CRS del DEM:")
print("x =", x_out)
print("y =", y_out)


## Ajuste del punto al río

Busca un punto de salida más adecuado para delimitar la cuenca.

Parte de un punto aproximado x, y y hace dos ajustes:

- Lo mueve al río principal más cercano del shapefile,
- Luego lo mueve al píxel cercano con mayor acumulación en el DEM.

La idea es evitar que el outlet quede:

- Fuera del río,
- Sobre un tributario pequeño,
- O entre píxeles que no representan bien el cauce

Esto mejora la delimitación de la cuenca.

In [ ]:
snapped_x, snapped_y, snapped_row, snapped_col = snap_to_high_accumulation(
    dem_lowres_path,
    acc,
    x_out,
    y_out,
    search_radius=40   # puedes probar 10, 20, 30
)

print("Outlet ajustado con acumulación:")
print("x =", snapped_x)
print("y =", snapped_y)
print("fila =", snapped_row)
print("columna =", snapped_col)

acc_value = acc[snapped_row, snapped_col]
acc_pct = 100 * acc_value / np.nanmax(acc)

print("acumulación =", acc_value)
print("porcentaje del máximo =", acc_pct)

## Visualización de la acumulación de flujo

La acumulación de flujo indica cuántas celdas aportan agua hacia cada píxel.

### Interpretación
- valores bajos: zonas de ladera o divisorias,
- valores altos: zonas donde se concentra el flujo, normalmente ríos o cauces.

Esta capa es muy útil para ajustar correctamente el punto de salida sobre la red de drenaje.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(
    np.log1p(acc),
    extent=grid.extent,
    cmap="Blues"
)

ax.set_title("Acumulación de flujo")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log(1 + acumulación)")

plt.show()

## Comparación entre el punto aproximado y el punto ajustado

El punto ingresado manualmente puede no coincidir exactamente con el cauce.

Por eso se ajusta al píxel cercano con mayor acumulación de flujo.

### En la figura:
- punto rojo: punto aproximado ingresado por el usuario,
- punto amarillo: punto ajustado a la red de drenaje.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(
    np.log1p(acc),
    extent=grid.extent,
    cmap="Blues"
)

ax.scatter(x_out, y_out, s=70, c="red", label="Punto aproximado")
ax.scatter(snapped_x, snapped_y, s=70, c="yellow", edgecolor="black", label="Punto ajustado")

ax.set_title("Ajuste del punto de salida")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend()

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log(1 + acumulación)")

plt.show()

## Delimitación de la cuenca

In [ ]:
catch = grid.catchment(
    x=snapped_col,
    y=snapped_row,
    fdir=fdir,
    xytype="index"
)

## Visualización de la cuenca delimitada

La delimitación genera una máscara raster donde se distinguen:

- celdas dentro de la cuenca,
- celdas fuera de la cuenca.

El punto rojo corresponde al exutorio o punto de salida.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

ax.imshow(catch, extent=grid.extent, cmap="Blues")
ax.scatter(snapped_x, snapped_y, s=70, c="red", label="Salida ajustada")

ax.set_title("Cuenca delimitada en formato raster")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend()

plt.show()

## Convertir a polígono

In [ ]:
mask = catch.astype(np.uint8)

results = shapes(mask, mask=mask > 0, transform=grid.affine)
geoms = [shape(g) for g, v in results if v == 1]

basin = gpd.GeoDataFrame(geometry=geoms, crs=grid.crs)
basin = basin.dissolve().explode(index_parts=False).reset_index(drop=True)

print("Polígono de cuenca generado correctamente.")

## Visualización del polígono de la cuenca

En esta figura se muestra el límite de la cuenca ya convertido a polígono.

Esto permite observar de forma más clara la forma de la cuenca y su relación con el punto de salida.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

basin.boundary.plot(ax=ax, color="blue", linewidth=2, label="Límite de cuenca")
ax.scatter(snapped_x, snapped_y, s=70, c="red", label="Punto de salida")

ax.set_title("Polígono de la cuenca delimitada")
ax.set_xlabel("Latitud")
ax.set_ylabel("Longitud")
ax.legend()

plt.show()

## Área y perímetro

In [ ]:
# =========================================================
# CALCULAR ÁREA Y PERÍMETRO
# =========================================================
# Si el CRS está en grados, reproyectamos a un CRS métrico para
# que área y perímetro salgan en unidades físicas correctas.

if basin.crs.is_geographic:
    basin_metric = basin.to_crs(basin.estimate_utm_crs())
else:
    basin_metric = basin.copy()

basin_geom = basin_metric.geometry.iloc[0]

# Área en metros cuadrados y km²
area_m2 = basin_geom.area
area_km2 = area_m2 / 1e6

# Perímetro en metros y km
perimeter_m = basin_geom.length
perimeter_km = perimeter_m / 1000

print(f"Área: {area_km2:.2f} km²")
print(f"Perímetro: {perimeter_km:.2f} km")

# Visualización de la cuenca hidrográfica

Esta visualización permite relacionar la forma de la cuenca con el relieve.

Es útil para interpretar:

- divisorias de aguas,
- extensión de la cuenca,
- relación entre topografía y drenaje.



In [ ]:
with rasterio.open(dem_lowres_path) as src:
    dem_plot = src.read(1).astype(float)

    if src.nodata is not None:
        dem_plot[dem_plot == src.nodata] = np.nan

    dem_plot = np.ma.masked_where(np.isnan(dem_plot), dem_plot)

    colors = [
        (0.85, 0.92, 0.75),
        (0.55, 0.75, 0.45),
        (0.85, 0.75, 0.55),
        (0.65, 0.45, 0.30),
        (0.95, 0.95, 0.95)
    ]

    cmap = LinearSegmentedColormap.from_list("custom_terrain", colors, N=256)

    fig, ax = plt.subplots(figsize=(10, 8))

    colors = [
        (0.85, 0.92, 0.75),
        (0.55, 0.75, 0.45),
        (0.85, 0.75, 0.55),
        (0.65, 0.45, 0.30),
        (0.95, 0.95, 0.95)
    ]

    cmap = LinearSegmentedColormap.from_list("custom_terrain", colors, N=256)

    im = ax.imshow(
        dem_plot,
        cmap=cmap,
        extent=(src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
    )

    basin.boundary.plot(ax=ax, color="black", linewidth=2)
    ax.scatter(snapped_x, snapped_y, s=80, c="red", label="Punto de salida")

    ax.set_title("Mapa final de la cuenca delimitada")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend()

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Elevación (m)")

    plt.show()

## Cálculo de atributos físicos de la cuenca

Una vez delimitada la cuenca, podemos calcular varios atributos físicos importantes.

### Ejemplos
- área,
- perímetro,
- elevación mínima, máxima y media,
- pendiente,
- longitud máxima aproximada,
- ancho medio aproximado,
- índices morfométricos.

Estos atributos son muy útiles para caracterizar hidrológicamente una cuenca.

In [ ]:
# =========================================================
# CALCULAR ELEVACIÓN Y PENDIENTE DENTRO DE LA CUENCA
# =========================================================
# Leemos el DEM y calculamos estadísticas solo sobre las celdas
# que pertenecen a la cuenca.

with rasterio.open(dem_lowres_path) as src:
    dem = src.read(1).astype(float)

    # Guardar resolución del raster
    res_x, res_y = src.res

    # Reemplazar NoData por NaN
    if src.nodata is not None:
        dem[dem == src.nodata] = np.nan

# Dejar solo valores de elevación dentro de la cuenca
dem_basin = np.where(catch > 0, dem, np.nan)

# Estadísticas de elevación
elev_min = np.nanmin(dem_basin)
elev_max = np.nanmax(dem_basin)
elev_mean = np.nanmean(dem_basin)
elev_std = np.nanstd(dem_basin)

print(f"Elevación mínima: {elev_min:.2f} m")
print(f"Elevación máxima: {elev_max:.2f} m")
print(f"Elevación media: {elev_mean:.2f} m")
print(f"Desviación estándar de elevación: {elev_std:.2f} m")

# ---------------------------------------------------------
# Calcular pendiente a partir del gradiente del DEM
# ---------------------------------------------------------
# np.gradient necesita el tamaño de pixel en x e y.
# Usamos abs() porque algunas resoluciones pueden venir negativas.

dx = abs(res_x)
dy = abs(res_y)

grad_y, grad_x = np.gradient(dem, dy, dx)
slope_rad = np.arctan(np.sqrt(grad_x**2 + grad_y**2))
slope_deg = np.degrees(slope_rad)

# Dejar solo pendiente dentro de la cuenca
slope_basin = np.where(catch > 0, slope_deg, np.nan)

# Estadísticas de pendiente
slope_mean = np.nanmean(slope_basin)
slope_min = np.nanmin(slope_basin)
slope_max = np.nanmax(slope_basin)

print(f"Pendiente media: {slope_mean:.2f} grados")
print(f"Pendiente mínima: {slope_min:.2f} grados")
print(f"Pendiente máxima: {slope_max:.2f} grados")

In [ ]:
# =========================================================
# CALCULAR LONGITUD MÁXIMA Y ANCHO MEDIO APROXIMADOS
# =========================================================
# Usamos el bounding box del polígono como aproximación simple.

minx, miny, maxx, maxy = basin_geom.bounds

# Longitud máxima aproximada: lado mayor del bounding box
length_max_m = max(maxx - minx, maxy - miny)
length_max_km = length_max_m / 1000

# Ancho medio aproximado = área / longitud
mean_width_m = area_m2 / length_max_m
mean_width_km = mean_width_m / 1000

print(f"Longitud máxima aproximada: {length_max_km:.2f} km")
print(f"Ancho medio aproximado: {mean_width_km:.2f} km")

In [ ]:
# =========================================================
# CALCULAR ÍNDICES MORFOMÉTRICOS
# =========================================================

# Factor de forma
form_factor = area_m2 / (length_max_m ** 2)

# Índice de compacidad de Gravelius
gravelius = perimeter_m / (2 * np.sqrt(np.pi * area_m2))

# Índice de circularidad
circularity = (4 * np.pi * area_m2) / (perimeter_m ** 2)

print(f"Factor de forma: {form_factor:.3f}")
print(f"Índice de compacidad: {gravelius:.3f}")
print(f"Índice de circularidad: {circularity:.3f}")

## Tabla resumen de atributos físicos

La siguiente tabla resume los principales parámetros físicos de la cuenca.

Esta tabla puede exportarse a un archivo CSV para usarla posteriormente en informes, prácticas o análisis complementarios.

In [ ]:
# =========================================================
# CREAR TABLA RESUMEN
# =========================================================

basin_summary = pd.DataFrame({
    "Parámetro": [
        "Área",
        "Perímetro",
        "Elevación mínima",
        "Elevación máxima",
        "Elevación media",
        "Desviación estándar de elevación",
        "Pendiente media",
        "Pendiente mínima",
        "Pendiente máxima",
        "Longitud máxima aproximada",
        "Ancho medio aproximado",
        "Factor de forma",
        "Índice de compacidad",
        "Índice de circularidad"
    ],
    "Valor": [
        area_km2,
        perimeter_km,
        elev_min,
        elev_max,
        elev_mean,
        elev_std,
        slope_mean,
        slope_min,
        slope_max,
        length_max_km,
        mean_width_km,
        form_factor,
        gravelius,
        circularity
    ],
    "Unidad": [
        "km²",
        "km",
        "m",
        "m",
        "m",
        "m",
        "grados",
        "grados",
        "grados",
        "km",
        "km",
        "-",
        "-",
        "-"
    ]
})

basin_summary

In [ ]:
output_csv = "Data/Ñuble/Caracteristicas_cuenca.csv"
basin_summary.to_csv(output_csv, index=False)
print("Tabla de atributos guardada en:", output_csv)

## Conclusiones

En este ejercicio aprendimos a:

- Trabajar con DEM  
- Preparar datos hidrológicos  
- Delimitar cuencas  
- Calcular parámetros básicos  
